# 유니버스 생성 → 전략 시그널

저장된 S&P 500 일봉으로 블록 인스턴스를 만들고, 실행기로 한 번 실행해서
단계별 프레임과 전략 시그널을 직접 확인한다.

커널은 `.venv/bin/python`을 고른다.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ai_studio").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from ai_studio.n02_schema import StrategyEdge, StrategyGraph, StrategyNode
from ai_studio.n03_market_data import DATA_PATH, load_ohlc
from ai_studio.n08_blocks import history_from_bars
from ai_studio.n12_strategies import momentum_top10
from ai_studio.n13_connect import validate_graph
from ai_studio.n14_engine import ExecutionContext, execute, execution_order

pd.set_option("display.width", 160)
print(ROOT)

/Users/dahamlee/projects/ai-studio


## 1. 시세 적재

일봉은 블록 사이로 흐르는 프레임이 아니라 실행 맥락이다. `ExecutionContext`로 넘긴다.

In [2]:
bars = load_ohlc()
history = history_from_bars(bars)
as_of = max(bar.date for bar in bars)

print(f"파일: {DATA_PATH}")
print(f"행 수: {len(bars):,}  종목 수: {len(history):,}")
print(f"기간: {min(bar.date for bar in bars)} ~ {as_of}")

pd.DataFrame(bars[:5])

파일: /Users/dahamlee/projects/ai-studio/data/sp500_ohlc_1y.csv
행 수: 125,892  종목 수: 503
기간: 2025-09-30 ~ 2026-09-29


,date,symbol,open,high,low,close,volume
0,2025-09-30,A,123.419998,128.389999,123.010002,128.350006,2417900.0
1,2025-10-01,A,128.339996,138.979996,128.190002,138.580002,4333300.0
2,2025-10-02,A,137.550003,140.639999,136.899994,138.699997,2228800.0
3,2025-10-03,A,139.000000,143.089996,138.850006,141.639999,2346000.0
4,2025-10-06,A,141.509995,141.940002,139.899994,141.610001,1538400.0


## 2. 블록 인스턴스 정의

노드 하나가 화면의 블록 하나다. `id`는 인스턴스 이름, `function_id`는 구현체,
세 번째 값은 그 인스턴스의 파라미터다. 같은 구현체를 두 번 놓으면 인스턴스가 두 개다.
아래에서 `oversold`와 `strong`은 둘 다 `condition.threshold`이고 임계값만 다르다.

In [3]:
nodes = (
    StrategyNode("universe", "universe.compose", {"market": "US_EQUITY", "benchmark": "SPX"}),
    StrategyNode("filter", "universe.exclude", {"minListingDays": 148}),
    StrategyNode("price", "universe.min-price", {"minClose": 1000}),
    StrategyNode("rsi", "indicator.rsi", {"period": 14}),
    StrategyNode("oversold", "condition.threshold", {"operator": "lt", "threshold": 30}),
    StrategyNode("strong", "condition.threshold", {"operator": "gt", "threshold": 50}),
    StrategyNode("trend", "momentum.trend", {"method": "price_above_ma", "slowDays": 60}),
    StrategyNode("entry_rule", "logic.combine", {"operator": "and"}),
    StrategyNode("cross", "momentum.ma-cross", {"fastDays": 20, "slowDays": 60}),
    StrategyNode("state_decision", "strategy.state", {}),
    StrategyNode("event_decision", "strategy.event", {}),
    StrategyNode("hybrid_decision", "strategy.hybrid", {}),
)

edges = (
    StrategyEdge("universe", "universe", "filter", "universe"),
    StrategyEdge("filter", "universe", "price", "universe"),
    StrategyEdge("price", "universe", "rsi", "universe"),
    StrategyEdge("rsi", "value", "oversold", "value"),
    StrategyEdge("rsi", "value", "strong", "value"),
    StrategyEdge("price", "universe", "trend", "universe"),
    StrategyEdge("trend", "condition", "entry_rule", "conditions"),
    StrategyEdge("strong", "condition", "entry_rule", "conditions"),
    StrategyEdge("price", "universe", "cross", "universe"),
    StrategyEdge("price", "universe", "state_decision", "universe"),
    StrategyEdge("entry_rule", "condition", "state_decision", "condition"),
    StrategyEdge("price", "universe", "event_decision", "universe"),
    StrategyEdge("cross", "entry", "event_decision", "entry"),
    StrategyEdge("cross", "exit", "event_decision", "exit"),
    StrategyEdge("price", "universe", "hybrid_decision", "universe"),
    StrategyEdge("entry_rule", "condition", "hybrid_decision", "eligible"),
    StrategyEdge("cross", "entry", "hybrid_decision", "entry"),
    StrategyEdge("cross", "exit", "hybrid_decision", "exit"),
)

graph = StrategyGraph(nodes, edges)
check = validate_graph(graph)
print(f"연결 검사: {check.ok}  {check.reason or ''}")
print("실행 순서:", " → ".join(node.id for node in execution_order(graph)))

연결 검사: True  
실행 순서: universe → filter → price → cross → event_decision → rsi → oversold → strong → trend → entry_rule → hybrid_decision → state_decision


## 3. 실행

`execute`는 연결을 검사하고 위상 순서대로 각 인스턴스를 부른다.
결과는 노드 id와 출력 포트로 꺼낸다.

In [4]:
context = ExecutionContext(as_of, history)
result = execute(graph, context)

pd.DataFrame(
    [
        {
            "node": node_id,
            "port": port,
            "frame": type(value).__name__,
            "rows": len(getattr(value, "rows", ()) or ()),
        }
        for node_id, ports in result.outputs.items()
        for port, value in ports.items()
    ]
)

,node,port,frame,rows
0,universe,universe,UniverseFrame,503
1,filter,universe,UniverseFrame,501
2,price,universe,UniverseFrame,13
3,cross,entry,EventFrame,23
4,cross,exit,EventFrame,28
5,event_decision,result,StrategyResult,0
6,rsi,value,IndicatorFrame,13
7,oversold,condition,ConditionFrame,13
8,strong,condition,ConditionFrame,13
9,trend,condition,ConditionFrame,13


In [5]:
result

ExecutionResult(outputs={'universe': {'universe': UniverseFrame(as_of='2026-09-29', rows=(UniverseRow(instrument_id='A', eligible=True, metadata=None), UniverseRow(instrument_id='AAPL', eligible=True, metadata=None), UniverseRow(instrument_id='ABBV', eligible=True, metadata=None), UniverseRow(instrument_id='ABNB', eligible=True, metadata=None), UniverseRow(instrument_id='ABT', eligible=True, metadata=None), UniverseRow(instrument_id='ACGL', eligible=True, metadata=None), UniverseRow(instrument_id='ACN', eligible=True, metadata=None), UniverseRow(instrument_id='ADBE', eligible=True, metadata=None), UniverseRow(instrument_id='ADI', eligible=True, metadata=None), UniverseRow(instrument_id='ADM', eligible=True, metadata=None), UniverseRow(instrument_id='ADP', eligible=True, metadata=None), UniverseRow(instrument_id='ADSK', eligible=True, metadata=None), UniverseRow(instrument_id='AEE', eligible=True, metadata=None), UniverseRow(instrument_id='AEP', eligible=True, metadata=None), UniverseRo

## 4. 유니버스 — `UniverseFrame`

In [6]:
universe = result.port("universe", "universe")
eligible = result.port("filter", "universe")
removed = sorted(set(universe.symbols()) - set(eligible.symbols()))

print(f"기준일 {eligible.as_of}")
print(f"전체 {len(universe.symbols()):,} → 통과 {len(eligible.symbols()):,}  제외 {removed}")

pd.DataFrame(eligible.rows).head(10)

기준일 2026-09-29
전체 503 → 통과 501  제외 ['FDXF', 'HONA']


,instrument_id,eligible,metadata
0,A,True,None
1,AAPL,True,None
2,ABBV,True,None
3,ABNB,True,None
4,ABT,True,None
5,ACGL,True,None
6,ACN,True,None
7,ADBE,True,None
8,ADI,True,None
9,ADM,True,None


유니버스는 두 단계를 거친다. `filter`는 상장일수가 짧은 종목을 빼고,
`price`는 기준일 종가가 1,000달러보다 낮은 종목을 뺀다.
RSI·추세·교차·전략 판단은 모두 `price` 출력을 받는다.
행을 지우며 거르는 방식이라, 남은 행의 `eligible` 칸은 모두 참이다.

In [7]:
priced = result.port("price", "universe")
closes = {
    symbol: next(bar.close for bar in reversed(history[symbol]) if bar.date <= as_of)
    for symbol in priced.symbols()
}

print(f"상장일수 통과 {len(eligible.symbols()):,} → 종가 1,000달러 이상 {len(priced.symbols()):,}")
pd.DataFrame(
    [{"instrument_id": symbol, "close": closes[symbol]} for symbol in priced.symbols()]
)

상장일수 통과 501 → 종가 1,000달러 이상 13


,instrument_id,close
0,AZO,2875.969971
1,BLK,1067.939941
2,EQIX,1015.270020
3,FIX,1665.050049
4,GWW,1248.670044
5,LLY,1184.630005
6,MPWR,1355.989990
7,MTD,1537.329956
8,MU,1065.079956
9,NVR,6234.629883


## 5. 지표 — `IndicatorFrame`

숫자만 만든다. 좋고 나쁨이나 매수·매도는 정하지 않는다.

In [9]:
indicators = result.port("rsi", "value")
rsi_table = pd.DataFrame(indicators.rows).rename(columns={"value": "rsi"})

print(f"행 수 {len(rsi_table):,}")

rsi_table
# rsi_table.sort_values("rsi").head(10)

행 수 13


,instrument_id,rsi
0,AZO,46.860310
1,BLK,48.953889
2,EQIX,43.061853
3,FIX,56.129500
4,GWW,38.677006
5,LLY,75.095497
6,MPWR,65.403288
7,MTD,85.363246
8,MU,55.298953
9,NVR,52.395370


## 6. 조건 — `ConditionFrame`

지금 조건을 만족하는지 종목마다 참·거짓이 남는다. 거짓인 종목도 행이 사라지지 않는다.

In [10]:
conditions = {
    "oversold (RSI < 30)": result.port("oversold", "condition"),
    "strong (RSI > 50)": result.port("strong", "condition"),
    "trend (종가 > 60일선)": result.port("trend", "condition"),
    "entry_rule (trend AND strong)": result.port("entry_rule", "condition"),
}

summary = pd.DataFrame(
    [
        {
            "condition": name,
            "rows": len(frame.rows),
            "true": sum(row.matched for row in frame.rows),
            "false": sum(not row.matched for row in frame.rows),
        }
        for name, frame in conditions.items()
    ]
)
display(summary)

entry_rule = result.port("entry_rule", "condition")
pd.DataFrame(entry_rule.rows).head(10)

,condition,rows,true,false
0,oversold (RSI < 30),13,0,13
1,strong (RSI > 50),13,6,7
2,trend (종가 > 60일선),13,5,8
3,entry_rule (trend AND strong),13,4,9


,instrument_id,matched
0,AZO,False
1,BLK,False
2,EQIX,False
3,FIX,False
4,GWW,False
5,LLY,True
6,MPWR,True
7,MTD,True
8,MU,True
9,NVR,False


## 7. 이벤트 — `EventFrame`

교차가 일어난 날만 행이 생긴다. 조건과 달리 거짓 행이 없다.

In [11]:
entry_events = result.port("cross", "entry")
exit_events = result.port("cross", "exit")

events = pd.DataFrame(
    [*entry_events.rows, *exit_events.rows]
)
events["date"] = events["as_of"].dt.date.astype(str)

print(f"전체 기간 진입 {len(entry_events.rows):,}  청산 {len(exit_events.rows):,}")
events[events["date"] == as_of][["instrument_id", "date", "kind"]].sort_values(["kind", "instrument_id"])

전체 기간 진입 23  청산 28


,instrument_id,date,kind


## 8. 전략 판단 — `StrategyResult`

세 전략 블록 모두 출력은 같은 계약이다. 입력으로 무엇을 연결했는지가 동작을 가른다.

- `strategy.state`: 조건만 받는다. 매 평가 시점에 참인 종목을 다시 맞춘다.
- `strategy.event`: 진입·청산 이벤트를 받는다. 발생한 날에만 포지션을 바꾼다.
- `strategy.hybrid`: 조건으로 진입을 제한하고 이벤트로 사고판다.

In [12]:
decisions = {
    "strategy.state": result.port("state_decision", "result"),
    "strategy.event": result.port("event_decision", "result"),
    "strategy.hybrid": result.port("hybrid_decision", "result"),
}


def signal_table(decision):
    return pd.DataFrame(
        [
            {
                "symbol": row.symbol,
                "eligible": row.eligible,
                "action": row.action,
                "desired_state": row.desired_state,
                "target_weight": row.target_weight,
                "reason": ",".join(row.reason_codes),
            }
            for row in decision.signals
        ]
    )


pd.DataFrame(
    [
        {
            "block": name,
            "execution_type": decision.strategy_execution_type,
            "as_of": decision.as_of,
            "signals": len(decision.signals),
            "HOLD": sum(row.action == "HOLD" for row in decision.signals),
            "ENTER": sum(row.action == "ENTER" for row in decision.signals),
            "EXIT": sum(row.action == "EXIT" for row in decision.signals),
            "REBALANCE": sum(row.action == "REBALANCE" for row in decision.signals),
        }
        for name, decision in decisions.items()
    ]
)

,block,execution_type,as_of,signals,HOLD,ENTER,EXIT,REBALANCE
0,strategy.state,STATE_REBALANCE,2026-09-29 21:00:00+00:00,13,0,0,0,13
1,strategy.event,EVENT_LIFECYCLE,2026-09-29 21:00:00+00:00,13,13,0,0,0
2,strategy.hybrid,HYBRID,2026-09-29 21:00:00+00:00,13,13,0,0,0


In [ ]:
state_signals = signal_table(decisions["strategy.state"])
held = state_signals[state_signals["target_weight"].fillna(0) > 0]

print(f"조건이 참인 {len(held)}종목을 같은 비중으로 담는다. 나머지는 목표 비중 0이다.")
held.head(10)

In [ ]:
event_signals = signal_table(decisions["strategy.event"])
moved = event_signals[event_signals["action"].isin(["ENTER", "EXIT"])]

print(f"기준일에 상태가 바뀐 종목 {len(moved)}개. 나머지 {len(event_signals) - len(moved)}개는 HOLD다.")
moved

## 9. 이벤트 전략과 혼합 전략의 차이

같은 교차 이벤트를 넣어도, 혼합 전략은 `eligible` 조건이 거짓인 종목의 진입을 막는다.
기준일의 교차 종목을 두 전략에서 나란히 비교한다.

In [ ]:
hybrid_signals = signal_table(decisions["strategy.hybrid"])
candidates = sorted(
    {row.instrument_id for row in entry_events.rows if row.as_of.date().isoformat() == as_of}
    | {row.instrument_id for row in exit_events.rows if row.as_of.date().isoformat() == as_of}
)

comparison = (
    event_signals[event_signals["symbol"].isin(candidates)][["symbol", "action", "target_weight"]]
    .rename(columns={"action": "event_action", "target_weight": "event_weight"})
    .merge(
        hybrid_signals[hybrid_signals["symbol"].isin(candidates)][
            ["symbol", "eligible", "action", "target_weight"]
        ].rename(columns={"action": "hybrid_action", "target_weight": "hybrid_weight"}),
        on="symbol",
    )
)
comparison

진입 조건 인스턴스를 더 엄격한 것으로 바꿔 끼우면 결과가 달라진다.
`eligible` 입력을 `entry_rule` 대신 `oversold`에 연결하고 같은 실행기로 다시 돌린다.
실행기 코드는 손대지 않는다.

In [ ]:
strict_graph = StrategyGraph(
    nodes=(
        StrategyNode("universe", "universe.compose", {}),
        StrategyNode("filter", "universe.exclude", {"minListingDays": 148}),
        StrategyNode("rsi", "indicator.rsi", {"period": 14}),
        StrategyNode("oversold", "condition.threshold", {"operator": "lt", "threshold": 30}),
        StrategyNode("cross", "momentum.ma-cross", {"fastDays": 20, "slowDays": 60}),
        StrategyNode("strict_decision", "strategy.hybrid", {}),
    ),
    edges=(
        StrategyEdge("universe", "universe", "filter", "universe"),
        StrategyEdge("filter", "universe", "rsi", "universe"),
        StrategyEdge("rsi", "value", "oversold", "value"),
        StrategyEdge("filter", "universe", "cross", "universe"),
        StrategyEdge("filter", "universe", "strict_decision", "universe"),
        StrategyEdge("oversold", "condition", "strict_decision", "eligible"),
        StrategyEdge("cross", "entry", "strict_decision", "entry"),
        StrategyEdge("cross", "exit", "strict_decision", "exit"),
    ),
)

strict_signals = signal_table(execute(strict_graph, context).port("strict_decision", "result"))
print(f"느슨한 조건 ENTER {int((event_signals['action'] == 'ENTER').sum())}건")
print(f"엄격한 조건 ENTER {int((strict_signals['action'] == 'ENTER').sum())}건")
strict_signals[strict_signals["symbol"].isin(candidates)][
    ["symbol", "eligible", "action", "target_weight"]
]

## 10. 점수 경로 — `TargetPortfolio`

조건과 이벤트가 아니라 점수로 고르는 전략은 선정·비중·제약을 지나 목표 포트폴리오로 끝난다.
`momentum_top10()`은 이미 연결된 인스턴스 묶음이다.

In [ ]:
momentum = execute(momentum_top10(), context)
selection = momentum.port("pick", "selection")
target = momentum.port("target", "portfolio")

print(f"현금 비중 {target.cash_weight:.2%}")
pd.DataFrame(selection.rows).merge(
    pd.DataFrame(target.positions), on="instrument_id"
)

## 11. 날짜를 바꿔 실행 — 백테스트의 원리

`as_of`는 고정된 오늘이 아니라 `ExecutionContext`에 넘기는 값이다.
그래프는 그대로 두고 날짜만 바꿔 다시 실행하면 그 시점의 판단이 나온다.

`history` 전체를 넘겨도 미래가 섞이지 않는 이유는 블록이 `_upto`로
`as_of`까지만 잘라 쓰기 때문이다. 유니버스 제외 조건에 상장일수 148일이 걸려 있어
148번째 거래일부터 시작한다.

In [ ]:
trading_days = sorted({bar.date for bar in bars})
sample_days = trading_days[147::10]

print(f"전체 거래일 {len(trading_days)}일 중 {len(sample_days)}개 시점을 10거래일 간격으로 실행한다.")

timeline = pd.DataFrame(
    [
        {
            "as_of": day,
            "universe": len(run.port("filter", "universe").symbols()),
            "entry_rule_true": sum(row.matched for row in run.port("entry_rule", "condition").rows),
            "state_held": sum(
                (row.target_weight or 0) > 0
                for row in run.port("state_decision", "result").signals
            ),
            "event_ENTER": sum(
                row.action == "ENTER" for row in run.port("event_decision", "result").signals
            ),
            "event_EXIT": sum(
                row.action == "EXIT" for row in run.port("event_decision", "result").signals
            ),
            "hybrid_ENTER": sum(
                row.action == "ENTER" for row in run.port("hybrid_decision", "result").signals
            ),
        }
        for day, run in (
            (day, execute(graph, ExecutionContext(day, history))) for day in sample_days
        )
    ]
)
timeline

읽는 방법이 몇 가지 있다.

- `universe`가 499에서 501로 늘어난다. 상장일수 148일을 채운 종목이 날짜가 가면서 들어온다.
- `entry_rule_true`가 73에서 292까지 흔들린다. 조건은 매 시점 다시 계산되는 상태값이다.
- `hybrid_ENTER`가 `event_ENTER`보다 작거나 같다. 조건이 거짓인 종목의 진입을 혼합 전략이 막는다.
- `event_EXIT`가 전부 0이다. 보유 포지션을 `ExecutionContext`에 넣지 않았기 때문에 청산할 대상이 없다.
  실제 백테스트는 아래처럼 원장이 보유 상태를 이어서 넘겨 준다.

## 12. 원장까지 붙인 백테스트

`run_momentum_topn()`은 같은 구조를 끝까지 돌린다. 날짜마다 그래프를 실행해 목표 비중을 얻고,
그날 종가를 기준가로 주문을 만들어 다음 거래일 시가에 체결한다.

In [13]:
from ai_studio.n15_ohlc_backtest import run_momentum_topn

backtest = run_momentum_topn(bars, lookback=20, top_n=10, step=21, cash_weight=0.2)
curve = pd.DataFrame(backtest.portfolio_history)
last = backtest.portfolio_history[-1]

print(f"리밸런싱 {len({row.as_of for row in backtest.signal_history})}회  체결 {len(backtest.trade_log)}건")
print(f"최종 평가액 {last.total_value:,.0f}  누적 수익률 {last.cumulative_return:.2%}")
print(f"최대 낙폭 {min(row.drawdown for row in backtest.portfolio_history):.2%}")

curve[["as_of", "cash", "securities_value", "total_value", "period_return", "cumulative_return", "drawdown"]]

리밸런싱 11회  체결 201건
최종 평가액 1,675,303  누적 수익률 67.53%
최대 낙폭 -8.88%


,as_of,cash,securities_value,total_value,period_return,cumulative_return,drawdown
0,2025-10-30 21:00:00+00:00,217020.908844,7.777682e+05,9.947891e+05,-0.005211,-0.005211,-0.005211
1,2025-12-01 21:00:00+00:00,198532.419876,8.350159e+05,1.033548e+06,0.038962,0.033548,0.000000
2,2025-12-31 21:00:00+00:00,212371.498375,8.364132e+05,1.048785e+06,0.014742,0.048785,0.000000
3,2026-02-02 21:00:00+00:00,238141.965410,9.489098e+05,1.187052e+06,0.131835,0.187052,0.000000
4,2026-03-04 21:00:00+00:00,236994.233843,9.207191e+05,1.157713e+06,-0.024715,0.157713,-0.024715
5,2026-04-02 21:00:00+00:00,213873.582861,9.498156e+05,1.163689e+06,0.005162,0.163689,-0.019681
6,2026-05-04 21:00:00+00:00,283629.396318,1.125649e+06,1.409278e+06,0.211043,0.409278,0.000000
7,2026-06-03 21:00:00+00:00,403853.978960,1.409038e+06,1.812892e+06,0.286397,0.812892,0.000000
8,2026-07-06 21:00:00+00:00,347769.662409,1.359547e+06,1.707317e+06,-0.058236,0.707317,-0.058236
9,2026-08-04 21:00:00+00:00,335286.149069,1.316620e+06,1.651906e+06,-0.032455,0.651906,-0.088801


In [14]:
last_day = max(row.as_of for row in backtest.position_history)
holdings = pd.DataFrame(
    [row for row in backtest.position_history if row.as_of == last_day and row.quantity]
)

for row in backtest.portfolio_history:
    bar = "█" * round(row.cumulative_return * 40)
    print(f"{row.as_of.date()}  {row.cumulative_return:+7.2%}  {bar}")

print(f"\n마지막 평가일 {last_day} 보유 {len(holdings)}종목")
holdings[["symbol", "quantity", "average_entry_price", "market_price", "market_value", "unrealized_pnl"]]

2025-10-30   -0.52%  
2025-12-01   +3.35%  █
2025-12-31   +4.88%  ██
2026-02-02  +18.71%  ███████
2026-03-04  +15.77%  ██████
2026-04-02  +16.37%  ███████
2026-05-04  +40.93%  ████████████████
2026-06-03  +81.29%  █████████████████████████████████
2026-07-06  +70.73%  ████████████████████████████
2026-08-04  +65.19%  ██████████████████████████
2026-09-02  +67.53%  ███████████████████████████

마지막 평가일 2026-09-02 21:00:00+00:00 보유 10종목


,symbol,quantity,average_entry_price,market_price,market_value,unrealized_pnl
0,CRM,520,259.380005,256.929993,133603.596191,-1274.006348
1,IT,706,189.070007,186.750000,131845.500000,-1637.925171
2,MPC,350,380.369995,387.000000,135450.000000,2320.501709
3,MRNA,870,151.399994,150.809998,131204.697876,-513.296814
4,NEM,1094,124.970001,125.160004,136925.044006,207.862671
5,NOW,939,140.300003,136.720001,128380.081146,-3361.621719
6,PLTR,746,176.990005,169.460007,126417.165009,-5617.379089
7,PSKY,12725,10.550000,10.970000,139593.253398,5344.500971
8,SMCI,3657,37.650002,37.000000,135309.000000,-2377.055580
9,VEEV,480,278.279999,280.720001,134745.600586,1171.201172
